## Example 1: astroid functionality test 1

In [ ]:
import astroid

# Example 1: Basic AST parsing and node creation
print("Example 1: Basic astroid AST parsing")

try:
    # Parse a simple Python code string
    code = """
def greet(name):
    message = f"Hello, {name}!"
    return message
    
result = greet("World")
print(result)
"""
    
    module = astroid.parse(code)
    
    print("✓ Code parsed successfully")
    print(f"Module type: {type(module).__name__}")
    print(f"Module name: {module.name}")
    
    # Get module body
    print(f"Module body contains {len(module.body)} statements")
    
    # Examine first statement (function definition)
    func_def = module.body[0]
    print(f"First statement: {type(func_def).__name__}")
    print(f"Function name: {func_def.name}")
    print(f"Function args: {len(func_def.args.args)}")
    assert [type(node).__name__ for node in module.body] == ['FunctionDef', 'Assign', 'Expr']
    assert func_def.name == 'greet' and [arg.name for arg in func_def.args.args] == ['name']
    
    # Inference follows the call into greet() and evaluates the f-string
    inferred = next(module.body[1].value.infer())
    print(f"Inferred value of result: {inferred.value!r}")
    assert inferred.value == "Hello, World!", inferred
    
    print("astroid example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"astroid example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: astroid functionality test 2

In [ ]:
import astroid

# Example 2: AST node traversal and inspection
print("Example 2: AST node traversal and inspection")

try:
    # Parse a more complex code example
    code = """
class Calculator:
    def __init__(self, name="calc"):
        self.name = name
        self.history = []
    
    def add(self, x, y):
        result = x + y
        self.history.append(('add', x, y, result))
        return result
        
calc = Calculator("MyCalc")
answer = calc.add(10, 20)
"""
    
    module = astroid.parse(code)
    print("✓ Complex code parsed")
    
    # Walk through all nodes
    node_types = {}
    for node in module.nodes_of_class(astroid.NodeNG):
        node_type = type(node).__name__
        node_types[node_type] = node_types.get(node_type, 0) + 1
    
    print(f"Node type distribution:")
    for node_type, count in sorted(node_types.items()):
        print(f"  {node_type}: {count}")
    
    # Find specific node types
    classes = list(module.nodes_of_class(astroid.ClassDef))
    functions = list(module.nodes_of_class(astroid.FunctionDef))
    
    print(f"\nFound {len(classes)} classes and {len(functions)} functions")
    
    assert len(classes) == 1 and [f.name for f in functions] == ['__init__', 'add'], (classes, functions)
    cls = classes[0]
    print(f"Class '{cls.name}' has {len(cls.body)} members")
    assert cls.name == 'Calculator' and len(cls.body) == 2
    assert node_types['ClassDef'] == 1 and node_types['FunctionDef'] == 2, node_types
    
    # Inference: calc is a Calculator instance and calc.add(10, 20) evaluates to 30
    calc_value = next(module['calc'].infer())
    answer_value = next(module['answer'].infer())
    print(f"calc infers to an instance of {calc_value.name}; answer infers to {answer_value.value}")
    assert calc_value.name == 'Calculator' and answer_value.value == 30
    
    print("astroid example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"astroid example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: astroid functionality test 3

In [ ]:
import astroid
from astroid import manager

# Example 3: Module loading and inference
print("Example 3: Module loading and inference")

try:
    # Create astroid manager for module loading
    mgr = manager.AstroidManager()
    
    print("✓ AstroidManager created")
    
    # Parse code with variables and assignments
    code = """
x = 42
y = "hello"
z = [1, 2, 3, 4, 5]
result = len(z) + x

def process_data(data):
    if isinstance(data, list):
        return sum(data)
    return 0

output = process_data(z)
"""
    
    module = astroid.parse(code)
    print("✓ Code with variables parsed")
    
    # Find assignments and try to infer types
    assignments = list(module.nodes_of_class(astroid.Assign))
    print(f"Found {len(assignments)} assignments")
    assert len(assignments) == 5, len(assignments)
    
    inferred_types = {}
    for assign in assignments[:3]:  # Check first 3 assignments
        if assign.targets and len(assign.targets) > 0:
            target = assign.targets[0]
            if hasattr(target, 'name'):
                try:
                    inferred = list(assign.value.infer())
                    if inferred:
                        inferred_type = type(inferred[0].value).__name__ if hasattr(inferred[0], 'value') else 'Unknown'
                        print(f"Variable '{target.name}': inferred type {inferred_type}")
                        inferred_types[target.name] = inferred_type
                except:
                    print(f"Variable '{target.name}': inference failed")
                    raise
    
    # Find function calls
    calls = list(module.nodes_of_class(astroid.Call))
    print(f"Found {len(calls)} function calls")
    assert inferred_types['x'] == 'int' and inferred_types['y'] == 'str', inferred_types
    assert len(calls) == 4, len(calls)  # len, isinstance, sum, process_data
    
    # len(z) + x is computed statically: 5 + 42
    result_value = next(module['result'].infer())
    print(f"Inferred value of result: {result_value.value}")
    assert result_value.value == 47, result_value
    
    print("astroid example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"astroid example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: astroid functionality test 4

In [ ]:
import astroid
import os

# Example 4: Real file analysis and code inspection
print("Example 4: Real file analysis and code inspection")

try:
    # Create a temporary Python file for analysis
    temp_code = '''
"""Sample module for astroid analysis."""

import sys
import os

class DataProcessor:
    """Processes various types of data."""
    
    def __init__(self, debug=False):
        self.debug = debug
        self.processed_count = 0
    
    def process_list(self, items):
        """Process a list of items."""
        if self.debug:
            print(f"Processing {len(items)} items")
        
        result = []
        for item in items:
            if isinstance(item, str):
                result.append(item.upper())
            elif isinstance(item, int):
                result.append(item * 2)
        
        self.processed_count += len(result)
        return result

def main():
    processor = DataProcessor(debug=True)
    data = ["hello", 42, "world", 7]
    result = processor.process_list(data)
    print(f"Result: {result}")

if __name__ == "__main__":
    main()
'''
    
    # Parse the module
    module = astroid.parse(temp_code, module_name="sample_module")
    print("✓ Sample module parsed")
    
    # Analyze imports
    imports = list(module.nodes_of_class(astroid.Import))
    import_froms = list(module.nodes_of_class(astroid.ImportFrom))
    print(f"Direct imports: {len(imports)}, From imports: {len(import_froms)}")
    assert (len(imports), len(import_froms)) == (2, 0)
    assert [name for node in imports for name, _ in node.names] == ['sys', 'os']
    
    # Analyze classes
    classes = list(module.nodes_of_class(astroid.ClassDef))
    for cls in classes:
        print(f"Class: {cls.name}")
        # Get docstring using the correct astroid API
        docstring = None
        if hasattr(cls, 'doc_node') and cls.doc_node:
            docstring = cls.doc_node.value
        elif hasattr(cls, 'get_doc_node'):
            doc_node = cls.get_doc_node()
            docstring = doc_node.value if doc_node else None
        
        print(f"  Docstring: {docstring[:50] + '...' if docstring else 'None'}")
        print(f"  Methods: {len([m for m in cls.body if isinstance(m, astroid.FunctionDef)])}")
    
    # Analyze functions
    functions = list(module.nodes_of_class(astroid.FunctionDef))
    standalone_functions = [f for f in functions if not isinstance(f.parent, astroid.ClassDef)]
    print(f"Standalone functions: {len(standalone_functions)}")
    assert [cls.name for cls in classes] == ['DataProcessor']
    assert docstring == 'Processes various types of data.', docstring
    assert [m.name for m in classes[0].body if isinstance(m, astroid.FunctionDef)] == ['__init__', 'process_list']
    assert [f.name for f in standalone_functions] == ['main']
    
    print("astroid example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"astroid example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: astroid functionality test 5

In [ ]:
import astroid
from importlib.metadata import version as distribution_version

# Example 5: astroid package capabilities and static analysis
print("Example 5: astroid package capabilities and static analysis")

try:
    print("astroid provides static analysis of Python code")
    
    # Package analysis
    attrs = [attr for attr in dir(astroid) if not attr.startswith('_')]
    print(f"Available components: {len(attrs)}")
    print(f"Key components: {attrs[:8]}")
    
    # Show version information
    print(f"astroid version: {astroid.__version__}")
    assert astroid.__version__ == distribution_version("astroid"), astroid.__version__
    
    # Demonstrate code quality analysis
    problem_code = '''
def problematic_function(x):
    # Various code issues for analysis
    unused_var = 42
    if x > 0:
        return x * 2
    elif x > 0:  # Unreachable condition
        return x * 3
    
    global_var = "should be local"  # pylint would catch this
    return None

class UnusedClass:
    pass

result = problematic_function(10)
'''
    
    module = astroid.parse(problem_code)
    print("✓ Problem code parsed for analysis")
    
    # Static analysis capabilities
    print("\nStatic analysis capabilities:")
    
    # Count different constructs
    analysis_results = {
        "Functions": len(list(module.nodes_of_class(astroid.FunctionDef))),
        "Classes": len(list(module.nodes_of_class(astroid.ClassDef))), 
        "Assignments": len(list(module.nodes_of_class(astroid.Assign))),
        "If statements": len(list(module.nodes_of_class(astroid.If))),
        "Comparisons": len(list(module.nodes_of_class(astroid.Compare))),
        "Return statements": len(list(module.nodes_of_class(astroid.Return)))
    }
    
    for construct, count in analysis_results.items():
        print(f"  {construct}: {count}")
    assert analysis_results == {
        "Functions": 1,
        "Classes": 1,
        "Assignments": 3,  # unused_var, global_var, result
        "If statements": 2,  # the elif is a nested If node
        "Comparisons": 2,
        "Return statements": 3
    }, analysis_results
    
    # Show astroid's use cases
    print("\nKey astroid capabilities:")
    capabilities = [
        "🔍 Static code analysis and AST parsing",
        "🐍 Python code understanding and inspection", 
        "🏗️ Foundation for tools like Pylint",
        "🔎 Type inference and code flow analysis",
        "📊 Code metrics and complexity analysis",
        "🔧 Refactoring and code transformation support",
        "🚨 Dead code and unreachable code detection",
        "📝 Documentation and code quality tools"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    print("\nastroid enables sophisticated Python static analysis")
    print("astroid example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"astroid example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")